In [29]:
import pandas as pd
import os
import requests
import numpy as np
import config
import subprocess
from io import StringIO
import soccerdata as sd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

In [30]:
GIT_REPO = os.path.expanduser("~/MyProjects/FPL/FPL-Elo-Insights")
subprocess.run(["git", "-C", GIT_REPO, "pull"], check=True)

def get_gw_state():

    
    return requests.get("https://draft.premierleague.com/api/game").json()
state = get_gw_state()
GW = state["next_event"]
LAST_GW = GW - 1
TOTAL_MINUTES = GW * 90

last_year_stats = pd.read_excel("seasons/2025-26/End2526_Full.xlsx")

FPL_DATA_FOLDER = os.path.expanduser('~/MyProjects/FPL/FPL-Elo-Insights/data/2026-2027')

fixtures_dir = os.path.join(FPL_DATA_FOLDER, "By Gameweek")
players = pd.read_csv(f"{FPL_DATA_FOLDER}/players.csv",encoding="utf-8")
teams = pd.read_csv(f"{FPL_DATA_FOLDER}/teams.csv",encoding="utf-8")

playerstats = pd.read_csv(f"{FPL_DATA_FOLDER}/By Gameweek/GW{LAST_GW}/playerstats.csv",encoding="utf-8")

From https://github.com/olbauday/FPL-Core-Insights
   c47416c51..75bee1dd7  main       -> origin/main


Updating c47416c51..75bee1dd7
Fast-forward
 data/2026-2027/By Gameweek/GW12/fixtures.csv       |    2 +-
 data/2026-2027/By Gameweek/GW12/matches.csv        |    2 +-
 data/2026-2027/By Gameweek/GW2/fixtures.csv        |    2 +-
 data/2026-2027/By Gameweek/GW2/matches.csv         |    2 +-
 data/2026-2027/By Gameweek/GW28/fixtures.csv       |    4 +-
 data/2026-2027/By Gameweek/GW28/matches.csv        |    4 +-
 data/2026-2027/By Gameweek/GW5/fixtures.csv        |    2 +-
 data/2026-2027/By Gameweek/GW5/matches.csv         |    2 +-
 .../By Gameweek/GW5/player_gameweek_stats.csv      | 1334 ++++++-------
 data/2026-2027/By Gameweek/GW5/playerstats.csv     | 1334 ++++++-------
 data/2026-2027/By Gameweek/GW6/fixtures.csv        |   12 +-
 data/2026-2027/By Gameweek/GW6/matches.csv         |   12 +-
 data/2026-2027/By Gameweek/GW6/momentum.csv        |   92 +
 .../By Gameweek/GW6/player_gameweek_stats.csv      |  667 +++++++
 .../2026-2027/By Gameweek/GW6/playermatchstats.csv |   40 +
 d

In [31]:
# 1. playerstats + players (get names, position, team_code)
merged = playerstats.merge(
    players,
    left_on="id",
    right_on="player_id",
    how="left",
    suffixes=("", "_p")      # keep playerstats names clean, suffix player dups
)

# 2. + teams (get team short_name)
merged = merged.merge(
    teams[["code", "short_name"]],
    left_on="team_code",
    right_on="code",
    how="left"
).rename(columns={"short_name": "team_name"}).drop(columns=["code"])


base_cols = [
    # --- identity ---
    "web_name",
    "position",
    "team_name",

    # --- core output / value ---
    "total_points",
    "points_per_game",
    "minutes",
    "starts",
    "bonus",
    "bps",
    "form",

    # --- expected (xG) — your model's backbone ---
    "expected_goals_per_90",
    "expected_assists_per_90",
    "expected_goal_involvements_per_90",
    "expected_goals_conceded_per_90",

    # --- defcon (your edge) ---
    "defensive_contribution",
    "defensive_contribution_per_90",
    "tackles",
    "clearances_blocks_interceptions",
    "recoveries",

    # --- set pieces (PL is set-piece heavy, you flagged this) ---
    "penalties_order",
    "corners_and_indirect_freekicks_order",
    "direct_freekicks_order",

    # --- goalkeeper-specific (if modeling GKs) ---
    "saves_per_90",
    "clean_sheets_per_90",
    "goals_conceded_per_90",

    # meta
    "player_id",
    "team_code",

    # --- availability (your status_multiplier layer) ---
    "status",
    "chance_of_playing_next_round"
]

base = merged[base_cols].copy()

base = base.sort_values(by='total_points', ascending=False)

In [39]:
all_fixtures = []

for folder in sorted(os.listdir(fixtures_dir)):
    if folder.startswith("GW"):
        gw_path = os.path.join(fixtures_dir, folder, "fixtures.csv")
        if os.path.exists(gw_path):
            df = pd.read_csv(gw_path, encoding="utf-8")
            all_fixtures.append(df)

fixtures_master = pd.concat(all_fixtures, ignore_index=True)
fixtures_master["tournament"] = fixtures_master["tournament"].fillna("")
fixtures_master = fixtures_master[
    fixtures_master["tournament"].str.lower().str.contains("prem")
]

# current ELO ratings for all clubs
gwtest = fixtures_master[fixtures_master["gameweek"] == 1].copy()
print(gwtest[["home_team","away_team", "home_team_elo","away_team_elo"]].to_string(index=False))


 home_team  away_team  home_team_elo  away_team_elo
       3.0        9.0            NaN            NaN
      94.0        6.0        1836.59        1777.31
      36.0        7.0        1842.01        1920.97
      17.0        2.0            NaN            NaN
      11.0       31.0        1803.43        1803.82
      88.0        1.0        1532.88        1915.32
      43.0       91.0        1970.85        1871.97
      40.0       56.0        1640.00        1735.93
       4.0       14.0        1837.53        1910.80
      54.0        8.0        1813.10        1831.10


In [19]:
# #### FIXTURE GAMBLER MACHINE LEARNING MODEL
played  = fixtures_master[fixtures_master["gameweek"] <= GW].copy()


elo_test = fixtures_master[fixtures_master["gameweek"] == GW].copy()

print("ELO nulls this GW:", elo_test["home_team_elo"].isnull().sum(), "/", len(elo_test))

# fixtures_master["elo_diff"] = fixtures_master["home_team_elo"] - fixtures_master["away_team_elo"]
# fixtures_master["goal_diff"] = fixtures_master["home_score"] - fixtures_master["away_score"]
# fixtures_master["home_win"] = (fixtures_master["goal_diff"] > 0).astype(int)

# ✅ Defensive stats
played["home_non_penalty_xg_allowed"] = played["away_non_penalty_xg"]
played["away_non_penalty_xg_allowed"] = played["home_non_penalty_xg"]

played["home_xg_set_play_allowed"] = played["away_xg_set_play"]
played["away_xg_set_play_allowed"] = played["home_xg_set_play"]

played["home_goals_conceded"] = played["away_score"]
played["away_goals_conceded"] = played["home_score"]

played["home_points"] = played["home_score"].gt(played["away_score"])*3 + \
                                 played["home_score"].eq(played["away_score"])*1
played["away_points"] = played["away_score"].gt(played["home_score"])*3 + \
                                 played["away_score"].eq(played["home_score"])*1

played["home_xg_on_target_xgot_allowed"] = played["away_xg_on_target_xgot"]
played["away_xg_on_target_xgot_allowed"] = played["home_xg_on_target_xgot"]

played["home_keeper_saves_allowed"] = played["away_keeper_saves"]
played["away_keeper_saves_allowed"] = played["home_keeper_saves"]

fixtures = played.copy()

### CREATING ROLLING METRICS

# ---- HOME ----
home = fixtures[[
    "gameweek",
    "home_team",
    "home_team_elo",
    "home_score",
    "home_goals_conceded",
    "home_points",
    "home_non_penalty_xg",
    "home_xg_set_play",
    "home_non_penalty_xg_allowed",
    "home_xg_set_play_allowed",
    "home_xg_on_target_xgot",
    "home_xg_on_target_xgot_allowed",
    "home_keeper_saves"
]].copy()

home.columns = [
    "gameweek",
    "team",
    "elo",
    "goals_scored",
    "goals_conceded",
    "points",
    "non_penalty_xg",
    "xg_set_play",
    "non_penalty_xg_allowed",
    "xg_set_play_allowed",
    "xg_on_target_xgot",
    "xg_on_target_xgot_allowed",
    "keeper_saves",
]
home["is_home"] = 1

# ---- AWAY ----
away = fixtures[[
    "gameweek",
    "away_team",
    "away_team_elo",
    "away_score",
    "away_goals_conceded",
    "away_points",
    "away_non_penalty_xg",
    "away_xg_set_play",
    "away_non_penalty_xg_allowed",
    "away_xg_set_play_allowed",
    "away_xg_on_target_xgot",
    "away_xg_on_target_xgot_allowed",
    "away_keeper_saves"
]].copy()

away.columns = [
    "gameweek",
    "team",
    "elo",
    "goals_scored",
    "goals_conceded",
    "points",
    "non_penalty_xg",
    "xg_set_play",
    "non_penalty_xg_allowed",
    "xg_set_play_allowed",
    "xg_on_target_xgot",
    "xg_on_target_xgot_allowed",
    "keeper_saves",
]
away["is_home"] = 0

team_fixtures = pd.concat([home, away], ignore_index=True)
team_fixtures = team_fixtures.sort_values(["team","gameweek"])

# Rolling set-play xG
team_fixtures["roll3_xg_set_play"] = (
    team_fixtures.groupby("team")["xg_set_play"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

# team_fixtures["roll5_xg_set_play"] = (
#     team_fixtures
#     .groupby("team")["xg_set_play"]
#     .rolling(window=5, min_periods=1)
#     .mean()
#     .reset_index(level=0, drop=True)
# )

# Rolling xG
team_fixtures["roll3_open_play_xg"] = (
    team_fixtures.groupby("team")["non_penalty_xg"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

# team_fixtures["roll5_open_play_xg"] = (
#     team_fixtures
#     .groupby("team")["non_penalty_xg"]
#     .rolling(window=5, min_periods=1)
#     .mean()
#     .reset_index(level=0, drop=True)
# )

## Rolling Defensive

# Rolling DEFENSIVE set-play xG allowed
team_fixtures["roll3_xg_set_play_allowed"] = (
    team_fixtures.groupby("team")["xg_set_play_allowed"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

# team_fixtures["roll5_xg_set_play_allowed"] = (
#     team_fixtures
#     .groupby("team")["xg_set_play_allowed"]
#     .rolling(window=5, min_periods=1)
#     .mean()
#     .reset_index(level=0, drop=True)
# )

# Rolling DEFENSIVE open-play xG allowed
team_fixtures["roll3_open_play_xg_allowed"] = (
    team_fixtures.groupby("team")["non_penalty_xg_allowed"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

# team_fixtures["roll5_open_play_xg_allowed"] = (
#     team_fixtures
#     .groupby("team")["non_penalty_xg_allowed"]
#     .rolling(window=5, min_periods=1)
#     .mean()
#     .reset_index(level=0, drop=True)
# )

# Rolling Actuals

# Finishing: are they turning xG into good shots?
team_fixtures["finishing_xgot_over"] = (
    team_fixtures["xg_on_target_xgot"] - team_fixtures["non_penalty_xg"]
)

# Defense / GK: are they conceding less/more than the xGOT they face?
team_fixtures["shot_stopping_diff"] = (
    team_fixtures["xg_on_target_xgot_allowed"] - team_fixtures["goals_conceded"]
)

# a simple keeper quality proxy
team_fixtures["keeper_intensity"] = (
    team_fixtures["keeper_saves"] /
    (team_fixtures["xg_on_target_xgot_allowed"].clip(lower=0.1))
)

# Keep rolling 

#add these when possible "roll5_finishing_over","roll5_shot_stopping", "roll5_keeper_intensity") and the below loop
# for col, new3 in [
#     ("finishing_xgot_over",  "roll3_finishing_over"),
#     ("shot_stopping_diff",   "roll3_shot_stopping"),
#     ("keeper_intensity",     "roll3_keeper_intensity"),
# ]:    
    # team_fixtures[new3] = (
    #     team_fixtures.groupby(["team", "is_home"])[col]
    #     .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    # )

    # team_fixtures[new5] = (
    #     team_fixtures
    #     .groupby(["team", "is_home"])[col]
    #     .rolling(window=5, min_periods=1)
    #     .mean()
    #     .reset_index(level=[0, 1], drop=True)
    # )

### WEIGHTED FORM

# add roll 5 back when possible

team_fixtures["weighted_set_play_strength"]  = team_fixtures["roll3_xg_set_play"]
team_fixtures["weighted_open_play_strength"] = team_fixtures["roll3_open_play_xg"]

#team defence
team_fixtures["weighted_set_play_defence"]   = team_fixtures["roll3_xg_set_play_allowed"]
team_fixtures["weighted_open_play_defence"]  = team_fixtures["roll3_open_play_xg_allowed"]




# team ATTACK power — xG only, no finishing (needs home/away split, defer to GW6+)
team_fixtures["weighted_attack"] = (
    0.60 * team_fixtures["roll3_open_play_xg"] +
    0.40 * team_fixtures["roll3_xg_set_play"]
)

# team DEFENCE power — xG-allowed only, no shot-stopping/keeper (same reason)
team_fixtures["weighted_defence"] = (
    -0.65 * team_fixtures["roll3_open_play_xg_allowed"] +
    -0.35 * team_fixtures["roll3_xg_set_play_allowed"]
)

# old team ATTACK power, to be restored after GW 6

# team_fixtures["weighted_attack"] = (
#     0.50 * team_fixtures["roll3_open_play_xg"] +
#     0.25 * team_fixtures["roll3_xg_set_play"] +
#     0.25 * team_fixtures["roll3_finishing_over"]
# )

# team_fixtures["weighted_defence"] = (
#     -0.50 * team_fixtures["roll3_open_play_xg_allowed"] +
#     -0.25 * team_fixtures["roll3_xg_set_play_allowed"] +
#      0.20 * team_fixtures["roll3_shot_stopping"] +
#      0.05 * team_fixtures["roll3_keeper_intensity"]
#)
# team_fixtures["weighted_attack"] = (
#     0.50 * team_fixtures["weighted_open_play_strength"] +
#     0.25 * team_fixtures["weighted_set_play_strength"] +
#     0.25 * (
#         0.7 * team_fixtures["roll3_finishing_over"] +
#         0.3 * team_fixtures["roll5_finishing_over"]
#     )
# # )
# # Weighted DEFENSIVE power
# team_fixtures["weighted_defence"] = (
#     -0.50 * (0.7 * team_fixtures["roll3_open_play_xg_allowed"] +
#             0.3 * team_fixtures["roll5_open_play_xg_allowed"]) +

#     -0.25 * (0.7 * team_fixtures["roll3_xg_set_play_allowed"] +
#             0.3 * team_fixtures["roll5_xg_set_play_allowed"]) +

#     0.20 * (0.7 * team_fixtures["roll3_shot_stopping"] +
#             0.3 * team_fixtures["roll5_shot_stopping"]) +

#     0.05 * (0.7 * team_fixtures["roll3_keeper_intensity"] +
#             0.3 * team_fixtures["roll5_keeper_intensity"])
# )

latest = team_fixtures.sort_values("gameweek").groupby("team").tail(1)
latest = latest.merge(teams[["code","short_name"]], left_on="team", right_on="code", how="left")
print(latest[["short_name","weighted_attack"]]
      .sort_values("weighted_attack", ascending=False)
      .to_string(index=False))

print(latest[["short_name","weighted_defence"]]
      .sort_values("weighted_defence", ascending=False)
      .to_string(index=False))

print(latest[["short_name", "team", "elo"]])
print(latest.columns.tolist())

ELO nulls this GW: 10 / 10
short_name  weighted_attack
       LEE         1.530667
       SUN         1.468000
       MCI         1.360000
       BHA         1.288000
       BRE         1.170667
       FUL         1.168000
       ARS         1.119333
       EVE         0.974667
       NFO         0.969333
       BOU         0.891333
       COV         0.874000
       TOT         0.870000
       CRY         0.838000
       AVL         0.819333
       LIV         0.809333
       IPS         0.806000
       HUL         0.801333
       MUN         0.748667
       NEW         0.533333
       CHE         0.488667
short_name  weighted_defence
       LIV         -0.520333
       EVE         -0.644833
       NFO         -0.661000
       BOU         -0.678000
       ARS         -0.724833
       TOT         -0.751833
       IPS         -0.786667
       HUL         -0.813333
       MUN         -0.905000
       BRE         -0.905333
       AVL         -1.079167
       LEE         -1.090833
       F

In [20]:
home_features = (
    team_fixtures[team_fixtures["is_home"] == 1]
    .rename(columns=lambda c: "home_" + c if c not in ["gameweek", "team", "is_home"] else c)
)

away_features = (
    team_fixtures[team_fixtures["is_home"] == 0]
    .rename(columns=lambda c: "away_" + c if c not in ["gameweek", "team", "is_home"] else c)
)

played = played.merge(
    home_features[["gameweek","team"] + [c for c in home_features.columns if c.startswith("home_")]],
    left_on=["gameweek","home_team"], right_on=["gameweek","team"], how="left"
).drop(columns=["team"])

played = played.merge(
    away_features[["gameweek","team"] + [c for c in away_features.columns if c.startswith("away_")]],
    left_on=["gameweek","away_team"], right_on=["gameweek","team"], how="left"
).drop(columns=["team"])

played["set_play_diff"] = (
    (played["home_weighted_set_play_strength"] -
     played["away_weighted_set_play_defence"]) -
    (played["away_weighted_set_play_strength"] -
     played["home_weighted_set_play_defence"])
)

played["open_play_diff"] = (
    (played["home_weighted_open_play_strength"] -
     played["away_weighted_open_play_defence"]) -
    (played["away_weighted_open_play_strength"] -
     played["home_weighted_open_play_defence"])
)

played["attack_diff"] = (
    (played["home_weighted_attack"] - played["away_weighted_defence"]) -
    (played["away_weighted_attack"] - played["home_weighted_defence"])
)

# played["elo_diff"] = (
#     played["home_elo"] -
#     played["away_elo"]
# )

# played["elo_context"] = played["elo_diff"] / 250

played = played.sort_values("gameweek")

train = played[played["gameweek"] < GW]
test  = played[played["gameweek"] == GW]

print(test.shape)                              

# later add ELO

FEATURES = ["attack_diff", "set_play_diff", "open_play_diff"] 


train = train.dropna(subset=FEATURES + ["home_score", "away_score"])

test  = test.dropna(subset=FEATURES)

X = train[FEATURES]
X_test = test[FEATURES]

print(X_test.head(20))

y_home = train["home_score"]
y_away = train["away_score"]

### FIXTURE PREDICTOR
model_home = LinearRegression().fit(X, y_home)
model_away = LinearRegression().fit(X, y_away)

train_home_pred = model_home.predict(X)
train_away_pred = model_away.predict(X)

mu_home  = train_home_pred.mean()
std_home = train_home_pred.std()

mu_away  = train_away_pred.mean()
std_away = train_away_pred.std()

test["pred_home_goals"] = model_home.predict(X_test)
test["pred_away_goals"] = model_away.predict(X_test)

test["pred_goal_diff"] = test["pred_home_goals"] - test["pred_away_goals"]

## SCORE NORMALIZATION
test["home_attack_fixture"] = test["pred_home_goals"] - mu_home
test["away_attack_fixture"] = test["pred_away_goals"] - mu_away

test["home_defence_fixture"] = mu_away - test["pred_away_goals"]
test["away_defence_fixture"] = mu_home - test["pred_home_goals"]

test = test.merge(
    teams[['code', 'short_name']],
    left_on='home_team',
    right_on='code',
    how='left'
).rename(columns={
    'short_name': 'home_team_name'
}).drop(columns=['code'])

test = test.merge(
    teams[['code', 'short_name']],
    left_on='away_team',
    right_on='code',
    how='left'
).rename(columns={
    'short_name': 'away_team_name'
}).drop(columns=['code'])

test["pred_total_goals"] = test["pred_home_goals"] + test["pred_away_goals"]
test["pred_total_goals"] = test["pred_total_goals"].round(1)

test["kickoff_time"] = pd.to_datetime(test["kickoff_time"])
test = test.sort_values("kickoff_time")

cols_order = ["home_team_name", "pred_home_goals", "pred_away_goals", "away_team_name",
                  "pred_total_goals", "pred_goal_diff", "attack_diff",
    "home_attack_fixture", "away_defence_fixture", "away_attack_fixture", "home_defence_fixture", 
    "home_weighted_attack", "away_weighted_defence", "away_weighted_attack", "home_weighted_defence",
   "set_play_diff", "open_play_diff", 
  "home_weighted_set_play_strength",     "home_weighted_open_play_strength", 
    "home_weighted_set_play_defence",     "home_weighted_open_play_defence",
     "away_weighted_set_play_strength",     "away_weighted_open_play_strength",
    "away_weighted_set_play_defence",   "away_weighted_open_play_defence"]


predictions = test[cols_order]

name_cols = ["home_team_name", "away_team_name"]
num_cols = [c for c in predictions.columns if c not in name_cols and pd.api.types.is_numeric_dtype(predictions[c])]

predictions = predictions.copy()
predictions[num_cols] = predictions[num_cols].round(1)

predictions.to_csv("predicted_fixtures_GW.csv", index=False)


(10, 179)
    attack_diff  set_play_diff  open_play_diff
50     0.281000       0.333333        0.443333
51    -1.313833       0.493333        0.480000
52    -0.274500      -0.190000        0.173333
53    -0.045333      -0.640000       -0.843333
54    -0.525167      -0.720000        0.153333
55     0.008333      -0.363333       -0.963333
56     0.488333      -0.216667       -0.116667
57    -0.939667       0.090000        0.976667
58    -0.341833       0.043333       -0.050000
59     0.527500      -0.956667       -2.040000


In [21]:
# ---- FIXTURE SCORE from predictor (single GW) ----
home_fx = test[["home_team","home_attack_fixture","home_defence_fixture","pred_away_goals"]].rename(
    columns={"home_team":"team_code","home_attack_fixture":"fixture_attack",
             "home_defence_fixture":"fixture_defence","pred_away_goals":"opp_pred_goals"})
away_fx = test[["away_team","away_attack_fixture","away_defence_fixture","pred_home_goals"]].rename(
    columns={"away_team":"team_code","away_attack_fixture":"fixture_attack",
             "away_defence_fixture":"fixture_defence","pred_home_goals":"opp_pred_goals"})
team_fx = pd.concat([home_fx, away_fx], ignore_index=True)

# Poisson clean-sheet probability — convex boost as opponent's predicted goals drop
team_fx["cs_prob"] = np.exp(-team_fx["opp_pred_goals"].clip(lower=0))

print(team_fx["cs_prob"].describe())
print(team_fx["cs_prob"].head(10))

# normalize
team_fx["fixture_attack_score"]  = team_fx["fixture_attack"] / (team_fx["fixture_attack"].abs().max() + 1e-9)
team_fx["fixture_defence_score"] = team_fx["cs_prob"] / (team_fx["cs_prob"].max() + 1e-9)



count    20.000000
mean      0.289130
std       0.161807
min       0.089130
25%       0.167790
50%       0.244167
75%       0.359700
max       0.699026
Name: cs_prob, dtype: float64
0    0.140243
1    0.271607
2    0.089130
3    0.199747
4    0.207484
5    0.169267
6    0.163361
7    0.230246
8    0.151923
9    0.317791
Name: cs_prob, dtype: float64


In [22]:
# 1. latest team strength (most recent rolling value per team)
teams_latest = (
    team_fixtures.sort_values("gameweek")
    .groupby("team").tail(1)[["team", "weighted_attack", "weighted_defence"]]
)

# 2. rescale to friendly 0-100 (linear — preserves order + distances)
def to_friendly(s, lo=0, hi=100):
    rng = s.max() - s.min()
    return pd.Series(50, index=s.index) if rng == 0 else lo + (s - s.min()) / rng * (hi - lo)

teams_latest["team_attack_score"]  = to_friendly(teams_latest["weighted_attack"])
teams_latest["team_defence_score"] = to_friendly(teams_latest["weighted_defence"])

# 3. merge onto player board (team_code ↔ team)
merged = base.merge(
    teams_latest[["team", "team_attack_score", "team_defence_score"]],
    left_on="team_code", right_on="team", how="left"
).drop(columns=["team"])

merged["team_score"] = np.select(
    [
        merged["position"] == "Forward",
        merged["position"] == "Midfielder",
        merged["position"] == "Defender",
        merged["position"] == "Goalkeeper",
    ],
    [
        merged["team_attack_score"],                                   # FWD: pure attack
        0.65 * merged["team_attack_score"] + 0.35 * merged["team_defence_score"],  # MID: mostly attack
        merged["team_defence_score"],                                  # DEF: pure defence
        merged["team_defence_score"],                                  # GK: pure defence
    ],
    default=0.0
)

merged = merged.merge(
    team_fx[["team_code","fixture_attack_score","fixture_defence_score"]],
    on="team_code", how="left"
)

merged["fixture_score"] = np.select(
    [merged["position"]=="Forward", merged["position"]=="Midfielder",
     merged["position"]=="Defender", merged["position"]=="Goalkeeper"],
    [merged["fixture_attack_score"],
     0.65*merged["fixture_attack_score"] + 0.35*merged["fixture_defence_score"],
     merged["fixture_defence_score"],
     merged["fixture_defence_score"]],
    default=0.0
)

merged["fixture_score_100"] = to_friendly(merged["fixture_score"].fillna(0))

print(merged.columns.tolist())
print(merged[["web_name", "team_name", "team_defence_score","team_attack_score", "team_score"]].head(20))

['web_name', 'position', 'team_name', 'total_points', 'points_per_game', 'minutes', 'starts', 'bonus', 'bps', 'form', 'expected_goals_per_90', 'expected_assists_per_90', 'expected_goal_involvements_per_90', 'expected_goals_conceded_per_90', 'defensive_contribution', 'defensive_contribution_per_90', 'tackles', 'clearances_blocks_interceptions', 'recoveries', 'penalties_order', 'corners_and_indirect_freekicks_order', 'direct_freekicks_order', 'saves_per_90', 'clean_sheets_per_90', 'goals_conceded_per_90', 'player_id', 'team_code', 'status', 'chance_of_playing_next_round', 'team_attack_score', 'team_defence_score', 'team_score', 'fixture_attack_score', 'fixture_defence_score', 'fixture_score', 'fixture_score_100']
       web_name team_name  team_defence_score  team_attack_score  team_score
0          Groß       BHA            4.436544          76.711452   51.415234
1     Tarkowski       EVE           88.452620          46.641075   88.452620
2         Bogle       LEE           47.086103   

In [23]:
###LAST SEASON'S STATS

# Columns to bring from last year
cols_needed = [
    "web_name",
    "points_per_game",
    "x_fpl_p90",
    "total_points",
    "bonus_p90",
    "minutes",
    "defcon_p90"
]


# Prepare last year slice with prefix
last_year_slim = last_year_stats[cols_needed].copy()
last_year_slim = last_year_slim.rename(columns={col: f"last_{col}" for col in cols_needed if col != "web_name"})
last_year_slim = last_year_slim[last_year_slim["last_total_points"] > 0]

last_year_slim = (
    last_year_slim
    .sort_values("last_minutes", ascending=False)
    .drop_duplicates("web_name", keep="first")
)

# Merge (current is LEFT, last year secondary)
merged = merged.merge(
    last_year_slim,
    left_on="web_name",
    right_on="web_name",
    how="left",
    indicator=True
)

In [24]:
def scale_global(df, col):
    s = df[col]
    median = s.median()
    iqr = (s.quantile(0.75) - s.quantile(0.25)) + 1e-6
    return (s - median) / iqr

merged["minutes_factor"] = np.sqrt((merged["minutes"] / (LAST_GW * 90)).clip(upper=1))

print(merged["minutes_factor"].describe())

def expected_fpl_p90(row):
    pos = row["position"]
    xg = row.get("expected_goals_per_90", 0)
    xa = row.get("expected_assists_per_90", 0)
    xgc = row.get("expected_goals_conceded_per_90", 0)
    saves = row.get("saves_per_90", 0)
    defcon = row.get("defensive_contribution_per_90", 0)

    base = 0

    # --- GOALS + ASSISTS (position-scaled) ---
    # if pos == "Goalkeeper":
    #     base = xg * 10 + xa * 3
    if pos == "Defender":
        base = xg * 6 + xa * 3
    elif pos == "Midfielder":
        base = xg * 5 + xa * 3
    elif pos == "Forward":
        base = xg * 4 + xa * 3

    # --- CLEAN SHEETS ---
    if pos == "Goalkeeper":
        clean_sheet_pts = row["clean_sheets_per_90"] * 5
    elif pos == "Defender":
        clean_sheet_pts = row["clean_sheets_per_90"] * 4
    elif pos == "Midfielder":
        clean_sheet_pts = row["clean_sheets_per_90"] * 1
    else:
        clean_sheet_pts = 0

    # --- GOALS CONCEDED  ---
    if pos in ["Goalkeeper", "Defender"]:
        conceded_penalty = -0.5 * xgc
    else:
        conceded_penalty = 0

    # --- SAVE POINTS (GK only: 1 per 3 saves) ---
    save_pts = 0
    if pos == "Goalkeeper":
        save_pts = saves / 3

    # --- DEFENSIVE BONUS ---
    if pos == "Defender":
        threshold = 10
    elif pos in ["Midfielder", "Forward"]:
        threshold = 12
    else:
        threshold = None
    if threshold is None:
        defcon_bonus = 0
    else:
        # convert per-90 defcon into expected per-match value
        defcon_match = defcon * row["minutes_factor"]
        
        # probability of hitting match-level threshold
        prob_hit = min(defcon_match / threshold, 1)
    
        # 2-point cap
        defcon_bonus = 2 * prob_hit    

    # ---- APPEARANCE POINTS ----
    mins_est = row["minutes_factor"] * 90

    if mins_est >= 60:
        appearance_pts = 2
    elif mins_est > 0:
        appearance_pts = 1
    else:
        appearance_pts = 0

    return (
        base +
        clean_sheet_pts +
        save_pts +
        defcon_bonus +
        conceded_penalty +
        appearance_pts
    )

POSITION_LIMITS = {"Goalkeeper": 16, "Defender": 40, "Midfielder": 40, "Forward": 24}

def compute_vorp_dynamic(df, metric=""):
    output = []

    for pos, limit in POSITION_LIMITS.items():
        subset = df[df["position"] == pos].copy()
        subset = subset.sort_values(metric, ascending=False)

        if len(subset) < limit:
            baseline = subset[metric].min()
        else:
            baseline = subset.iloc[limit - 1][metric]

        print(pos, len(subset), baseline)

        subset["baseline"] = baseline
        subset["VORP"] = subset[metric] - baseline
        subset["above_replacement"] = subset["VORP"] > 0

        output.append(subset)

    return pd.concat(output, ignore_index=True)


count    667.000000
mean       0.413076
std        0.396754
min        0.000000
25%        0.000000
50%        0.336650
75%        0.825967
max        1.000000
Name: minutes_factor, dtype: float64


In [25]:
MIN_MINUTES = 15   # expand as season progresses
merged = merged[merged["minutes"] >= MIN_MINUTES].copy()

merged["current_x_fpl_p90"] = merged.apply(expected_fpl_p90, axis=1).fillna(0)
merged["bps_GW"] = merged["bps"] / LAST_GW

for col in ["current_x_fpl_p90", "points_per_game", "bps_GW", "last_points_per_game", "team_score", "fixture_score"]:
    merged[col + "_scaled"] = scale_global(merged, col)

print(merged["fixture_score"].describe())

status_multiplier = {"a": 1.0, "d": 0.9, "i": 0.3, "s": 0.3}
merged["availability_factor"] = merged["status"].map(status_multiplier).fillna(1.0)

merged["set_piece_score"] = 0.0
merged.loc[merged["penalties_order"] == 1, "set_piece_score"] += 0.15
merged.loc[merged["direct_freekicks_order"] == 1, "set_piece_score"] += 0.08
merged.loc[merged["corners_and_indirect_freekicks_order"] == 1, "set_piece_score"] += 0.05

#TODO bake the set piece score into secondary position_score function

w_current = 0.7
w_last = 0.3

merged["overall_expected_return"] = (
    w_current * (
        0.43 * merged["current_x_fpl_p90_scaled"] +
        0.37 * merged["points_per_game_scaled"] +
        0.11 * merged["bps_GW_scaled"] +
        0.09 * merged["team_score_scaled"]
    ).fillna(0) +
    w_last * merged["last_points_per_game_scaled"].fillna(0)
)

merged["weekly_expected_return"] = (
    w_current * (
        0.34 * merged["current_x_fpl_p90_scaled"] +
        0.29 * merged["points_per_game_scaled"] +
        0.07 * merged["bps_GW_scaled"] +
        0.22 * merged["fixture_score_scaled"].fillna(0)+
        0.08 * merged["team_score_scaled"]
    ).fillna(0) +
    w_last * merged["last_points_per_game_scaled"].fillna(0)
)

# --- apply stability + availability modifiers ---
merged["weekly_expected_return"] = merged["weekly_expected_return"] * merged["minutes_factor"]
merged["overall_expected_return"] = merged["overall_expected_return"] * merged["minutes_factor"]

merged["weekly_expected_return"] = np.where(
    merged["weekly_expected_return"] > 0,
    merged["weekly_expected_return"] * merged["availability_factor"],   # positive: multiply (penalty works)
    merged["weekly_expected_return"]                                     # negative: leave as-is (don't flip)
)

merged["buy_low"] = merged["current_x_fpl_p90_scaled"] - merged["points_per_game_scaled"]
merged["sell_high"] = merged["points_per_game_scaled"] - merged["current_x_fpl_p90_scaled"]

print(merged["buy_low"].describe())

count    390.000000
mean       0.223592
std        0.375030
min       -0.999264
25%        0.004484
50%        0.257545
75%        0.405587
max        1.000000
Name: fixture_score, dtype: float64
count    390.000000
mean      -0.234391
std        0.594449
min       -1.787778
25%       -0.556600
50%       -0.253697
75%        0.068923
max        4.682894
Name: buy_low, dtype: float64


In [26]:
weekly = compute_vorp_dynamic(merged, metric="weekly_expected_return")[["player_id","VORP"]].rename(columns={"VORP":"weekly_VORP"})
overall = compute_vorp_dynamic(merged, metric="overall_expected_return")[["player_id","VORP"]].rename(columns={"VORP":"overall_VORP"})

board = merged.merge(weekly, on="player_id").merge(overall, on="player_id")
board = board.sort_values("overall_VORP", ascending=False)

Goalkeeper 24 -0.007116755003334224
Defender 141 0.2559785698438235
Midfielder 179 0.198870178937263
Forward 46 -0.09950756203778195
Goalkeeper 24 -0.08641456173761247
Defender 141 0.23074949814845921
Midfielder 179 0.27391852347107515
Forward 46 -0.05218387994932629


In [27]:
BASE_URL = "https://draft.premierleague.com/api"

names = ["Tzolakis","M.Sangaré","Yalcouyé","Araujo","Tzolis"]

all_boards = []

for key, league in config.LEAGUES.items():
    LEAGUE_ID = league["league_id"]
    MY_ENTRY_ID = league["entry_id"]
    league_name = league["name"]          # "Gegen" / "Dalot"

    url = f"{BASE_URL}/league/{LEAGUE_ID}/element-status"
    raw = requests.get(url).json()
    league_players = pd.json_normalize(raw["element_status"])

    league_players["owner"] = pd.to_numeric(league_players["owner"], errors="coerce").astype("Int64")
    league_players["is_available"] = league_players["status"].astype(str) == "a"
    league_players["is_mine"] = league_players["owner"] == MY_ENTRY_ID
    league_players["is_taken"] = (~league_players["is_available"]) & (~league_players["is_mine"])
    league_players = league_players.rename(columns={"element": "player_id"})

    mine_set  = league.get("known_mine", set())
    taken_set = league.get("known_taken", set())
    avail_set = league.get("known_available", set())
    
    # mine / taken first
    league_players.loc[league_players["player_id"].isin(mine_set),  "is_mine"] = True
    league_players.loc[league_players["player_id"].isin(taken_set), "is_taken"] = True
    league_players.loc[league_players["player_id"].isin(mine_set | taken_set), "is_available"] = False

    # known_available LAST — fully reset to available, override everything
    league_players.loc[league_players["player_id"].isin(avail_set), ["is_mine","is_taken"]] = False
    league_players.loc[league_players["player_id"].isin(avail_set), "is_available"] = True

    league_players["is_mine"] = league_players["is_mine"].fillna(False).astype(bool)
    league_players["is_taken"] = league_players["is_taken"].fillna(False).astype(bool)

    check = league_players[league_players["player_id"].isin({557,565,572,579,605,608})]
    print(config.LEAGUES["gegenpressure"].keys())
    print(league_name)
    print(check[["player_id","is_mine","is_available","is_taken"]].to_string(index=False))

    flags = league_players[["player_id", "is_mine", "is_available", "is_taken"]].rename(columns={
        "is_mine":      f"mine_{league_name}",
        "is_available": f"available_{league_name}",
        "is_taken":     f"taken_{league_name}",
    })
    all_boards.append(flags)



dict_keys(['name', 'entry_id', 'league_id', 'league_size', 'format', 'known_taken', 'known_mine', 'known_available'])
Gegen
 player_id  is_mine  is_available  is_taken
       572    False         False      True
       557    False         False      True
       565    False         False      True
       605     True         False     False
       579    False         False      True
       608    False          True     False
dict_keys(['name', 'entry_id', 'league_id', 'league_size', 'format', 'known_taken', 'known_mine', 'known_available'])
Dalot
 player_id  is_mine  is_available  is_taken
       557    False         False      True
       565    False         False      True
       572    False         False      True
       579    False          True     False
       605    False         False      True
       608    False          True     False


In [28]:
enriched = board.copy()
for flags in all_boards:
    enriched = enriched.merge(flags, on="player_id", how="left")

enriched["mins_per_gw"] = enriched["minutes"] / LAST_GW

enriched = enriched.rename(columns={
    "current_x_fpl_p90": "x_fpl_p90",   # what the HTML expects
    "defensive_contribution_per_90": "defcon_p90",
    "expected_goal_involvements_per_90": "xgi_p90"
})

print(enriched.columns.tolist())

enriched.to_csv("weekly_board_enriched.csv", index=False, encoding="utf-8-sig")


['web_name', 'position', 'team_name', 'total_points', 'points_per_game', 'minutes', 'starts', 'bonus', 'bps', 'form', 'expected_goals_per_90', 'expected_assists_per_90', 'xgi_p90', 'expected_goals_conceded_per_90', 'defensive_contribution', 'defcon_p90', 'tackles', 'clearances_blocks_interceptions', 'recoveries', 'penalties_order', 'corners_and_indirect_freekicks_order', 'direct_freekicks_order', 'saves_per_90', 'clean_sheets_per_90', 'goals_conceded_per_90', 'player_id', 'team_code', 'status', 'chance_of_playing_next_round', 'team_attack_score', 'team_defence_score', 'team_score', 'fixture_attack_score', 'fixture_defence_score', 'fixture_score', 'fixture_score_100', 'last_points_per_game', 'last_x_fpl_p90', 'last_total_points', 'last_bonus_p90', 'last_minutes', 'last_defcon_p90', '_merge', 'minutes_factor', 'x_fpl_p90', 'bps_GW', 'current_x_fpl_p90_scaled', 'points_per_game_scaled', 'bps_GW_scaled', 'last_points_per_game_scaled', 'team_score_scaled', 'fixture_score_scaled', 'availabil